# Qwen2.5-3B Sentiment Pipeline (Gold Commodity Eval)

This notebook runs three stages:
1. **Zero-shot baseline** with `Qwen/Qwen2.5-3B-Instruct` on the Gold Commodity dataset.
2. **qLoRA fine-tuning** of `Qwen/Qwen2.5-3B-Instruct` on full FinancialPhraseBank, then evaluation on Gold Commodity.
3. **Random-search hyperparameter tuning** for Qwen qLoRA, then final re-evaluation on Gold Commodity.

All stage metrics are logged locally for comparison without W&B.

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [2]:
%%capture
!pip install -U unsloth transformers==4.56.2 trl==0.22.2 peft bitsandbytes accelerate datasets scikit-learn matplotlib pandas python-dotenv

## Device Check


In [3]:
import subprocess
import torch

print("=== nvidia-smi (if available) ===")
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout.strip():
        print(res.stdout)
    else:
        print(res.stderr.strip() or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found on this machine.")

if torch.cuda.is_available():
    device_type = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"

print(f"Detected runtime device: {device_type}")
if device_type == "cuda":
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

=== nvidia-smi (if available) ===
Tue Apr  7 16:52:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.44.01              Driver Version: 590.44.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
| 30%   30C    P8             25W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

## 1. Setup and Reproducibility

In [4]:
import os
import re
import gc
import math
import time
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def load_dotenv_walk(start: Path) -> bool:
    for parent in [start, *start.parents]:
        env_file = parent / '.env'
        if env_file.exists():
            load_dotenv(env_file, override=False)
            print(f'Loaded .env from: {parent}')
            return True
    return False

_ = load_dotenv_walk(Path(os.getcwd()))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print(f'Device: {DEVICE}')
print(f'DType : {DTYPE}')

Device: cuda
DType : torch.bfloat16


## 2. W&B + Dataset Loading (Gold + FinancialPhraseBank)

In [5]:
from datasets import load_dataset, Dataset
import pandas as pd

# Load dataset
gold_ds = load_dataset("SaguaroCapital/sentiment-analysis-in-commodity-market-gold")
split_name = "test" if "test" in gold_ds else "train"
gold_raw_df = gold_ds[split_name].to_pandas()

# Text column
sentence_col = "News"

# One-hot label columns
label_cols = [
    "Price Direction Up",
    "Price Direction Constant",
    "Price Direction Down"
]

# Check expected columns exist
missing = [c for c in [sentence_col] + label_cols if c not in gold_raw_df.columns]
if missing:
    raise ValueError(
        f"Missing expected columns: {missing}. Available columns: {list(gold_raw_df.columns)}"
    )

# Convert one-hot labels into class names
def get_label(row):
    if row["Price Direction Up"] == 1:
        return "positive"
    elif row["Price Direction Constant"] == 1:
        return "neutral"
    elif row["Price Direction Down"] == 1:
        return "negative"
    else:
        return "unknown"

gold_raw_df["label_text"] = gold_raw_df.apply(get_label, axis=1)

# Final evaluation dataframe used by batched_generate_labels / evaluate_predictions
gold_eval_df = pd.DataFrame({
    "sentence": gold_raw_df[sentence_col].astype(str),
    "label_text": gold_raw_df["label_text"].astype(str)
})

# Optional: create a Hugging Face Dataset with a text column for trainer eval use
# This is only useful if you want to pass this dataset into SFTTrainer as eval_dataset
gold_raw_df["text"] = gold_raw_df[sentence_col].astype(str)
eval_ds = Dataset.from_pandas(gold_raw_df[["text"]], preserve_index=False)

print(f"Gold eval samples: {len(gold_eval_df)}")
print(gold_eval_df["label_text"].value_counts())
print(gold_eval_df.head())

print("\nTrainer eval dataset:")
print(eval_ds)
print(eval_ds.column_names)

/common/home/users/r/rohitht.2022/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gold eval samples: 2114
label_text
positive    881
negative    757
unknown     394
neutral      82
Name: count, dtype: int64
                                            sentence label_text
0  Gold / Silver / Copper futures - weekly outloo...    unknown
1  gold to be a safe-haven again; sell crude on r...    unknown
2  feb. gold settles at $1,097,90/oz on comex, do...   negative
3  dec gold rises 30c to $443.40/oz in morning ny...   positive
4    Gold holds modest losses after Chicago PMI miss   negative

Trainer eval dataset:
Dataset({
    features: ['text'],
    num_rows: 2114
})
['text']


In [6]:
# Download FinancialPhraseBank (75% agreement)
!curl -L "https://raw.githubusercontent.com/maxwellsarpong/NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt" -o Sentences_75Agree.txt

fpb_df = pd.read_csv(
    'Sentences_75Agree.txt',
    sep='@',
    header=None,
    names=['sentence', 'label_text'],
    engine='python',
    encoding='latin-1',
)
fpb_df['sentence'] = fpb_df['sentence'].str.strip()
fpb_df['label_text'] = fpb_df['label_text'].str.strip().str.lower()

print(f'FPB full samples: {len(fpb_df)}')
print(fpb_df['label_text'].value_counts())

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  452k  100  452k    0     0  1351k      0 --:--:-- --:--:-- --:--:-- 1355k
FPB full samples: 3453
label_text
neutral     2146
positive     887
negative     420
Name: count, dtype: int64


## 6. Random Search Tuning (Qwen qLoRA) + Final Evaluation on Gold

This stage uses a small random search over qLoRA hyperparameters with early stopping to keep tuning fast on GPU clusters.

In [7]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
)

LABELS = ['negative', 'neutral', 'positive']
LABEL_SET = set(LABELS)

def make_prompt(sentence: str) -> str:
    return (
        'You are a financial sentiment classifier.\n'
        'Return exactly one label: negative, neutral, or positive.\n\n'
        f'Sentence: {sentence}\n'
        'Label:'
    )

def parse_label(text: str) -> str:
    t = text.strip().lower()
    m = re.search(r'(negative|neutral|positive)', t)
    return m.group(1) if m else 'neutral'

@torch.inference_mode()
def batched_generate_labels(model, tokenizer, sentences, batch_size=32, max_new_tokens=3):
    preds = []
    tokenizer.padding_side = 'left'
    prompts = [make_prompt(s) for s in sentences]

    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i : i + batch_size]
        enc = tokenizer(
            chunk,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512,
        ).to(model.device)

        out = model.generate(
            **enc,
            do_sample=False,
            temperature=None,
            top_p=None,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

        gen_tokens = out[:, enc['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        preds.extend([parse_label(t) for t in texts])

    return preds

def evaluate_predictions(true_labels, pred_labels):
    true_labels = [x.strip().lower() for x in true_labels]
    pred_labels = [x.strip().lower() if x.strip().lower() in LABEL_SET else 'neutral' for x in pred_labels]

    metrics = {
        'macro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='macro', zero_division=0),
        'micro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='micro', zero_division=0),
        'weighted_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='weighted', zero_division=0),
        'f1_negative': f1_score(true_labels, pred_labels, labels=['negative'], average='macro', zero_division=0),
        'f1_neutral': f1_score(true_labels, pred_labels, labels=['neutral'], average='macro', zero_division=0),
        'f1_positive': f1_score(true_labels, pred_labels, labels=['positive'], average='macro', zero_division=0),
        'accuracy': accuracy_score(true_labels, pred_labels),
        'mcc': matthews_corrcoef(true_labels, pred_labels),
    }

    metrics['classification_report'] = classification_report(
        true_labels, pred_labels, labels=LABELS, zero_division=0
    )
    metrics['confusion_matrix'] = confusion_matrix(true_labels, pred_labels, labels=LABELS)
    return metrics

def print_metrics(title, metrics):
    print('=' * 80)
    print(title)
    print('=' * 80)
    print(metrics['classification_report'])
    print(f"Macro F1   : {metrics['macro_f1']:.4f}")
    print(f"Micro F1   : {metrics['micro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
    print(f"Accuracy   : {metrics['accuracy']:.4f}")
    print(f"MCC        : {metrics['mcc']:.4f}")
    print(f"F1 neg     : {metrics['f1_negative']:.4f}")
    print(f"F1 neu     : {metrics['f1_neutral']:.4f}")
    print(f"F1 pos     : {metrics['f1_positive']:.4f}")

def log_run_to_wandb(run_name, config_dict, metrics):
    # W&B logging has been disabled
    print(f"[W&B Disabled] Would have logged run: {run_name}")

## 4. Baseline: Qwen2.5-3B-Instruct (Non-Finetuned) on Gold Commodity

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, use_fast=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=DTYPE,
    device_map='auto',
    load_in_4bit=True,
)
qwen_model.eval()

qwen_preds = batched_generate_labels(
    qwen_model,
    qwen_tokenizer,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

qwen_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), qwen_preds)
print_metrics('Qwen2.5-3B-Instruct (Zero-shot) on Gold Commodity', qwen_metrics)

log_run_to_wandb(
    run_name='qwen2.5-3b-instruct_zero_shot_gold',
    config_dict={
        'model_name': QWEN_MODEL_NAME,
        'stage': 'baseline_zero_shot',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        'quantization': '4bit',
        'batch_size': 64,
    },
    metrics=qwen_metrics,
)

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.23s/it]
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen2.5-3B-Instruct (Zero-shot) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.75      0.64      0.69       757
     neutral       0.06      0.94      0.11        82
    positive       0.76      0.12      0.20       881

   micro avg       0.32      0.39      0.35      1720
   macro avg       0.53      0.57      0.34      1720
weighted avg       0.73      0.39      0.42      1720

Macro F1   : 0.3357
Micro F1   : 0.3479
Weighted F1: 0.4153
Accuracy   : 0.3155
MCC        : 0.2684
F1 neg     : 0.6938
F1 neu     : 0.1088
F1 pos     : 0.2045
[W&B Disabled] Would have logged run: qwen2.5-3b-instruct_zero_shot_gold


## 5. Fine-tune LLaMA3.2-3B-Instruct (Full FPB), Then Evaluate on Gold

In [9]:
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

LLAMA_MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'
LLAMA_ADAPTER_DIR = 'qlora_adapters/llama3.2-3b-fpb-full-default'
MAX_SEQ_LEN = 1024

def make_chat_train_text(tokenizer, sentence, label):
    messages = [
        {
            'role': 'system',
            'content': 'You are a financial sentiment classifier. Reply with one word: negative, neutral, or positive.',
        },
        {'role': 'user', 'content': f'Classify sentiment:\n{sentence}'},
        {'role': 'assistant', 'content': label},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

llama_model, llama_tokenizer = FastLanguageModel.from_pretrained(
    model_name=LLAMA_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

llama_model = FastLanguageModel.get_peft_model(
    llama_model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

train_ds = Dataset.from_dict({
    'text': [make_chat_train_text(llama_tokenizer, s, y) for s, y in zip(fpb_df['sentence'], fpb_df['label_text'])]
})

trainer = SFTTrainer(
    model=llama_model,
    tokenizer=llama_tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    data_collator=DataCollatorForSeq2Seq(tokenizer=llama_tokenizer),
    packing=False,
    args=SFTConfig(
        output_dir='outputs/llama3.2-3b-fpb-full-default',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim='adamw_8bit',
        logging_steps=20,
        save_strategy='epoch',
        seed=SEED,
        report_to='none',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
    response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
)

train_out = trainer.train()
llama_model.save_pretrained(LLAMA_ADAPTER_DIR)
llama_tokenizer.save_pretrained(LLAMA_ADAPTER_DIR)

FastLanguageModel.for_inference(llama_model)
llama_model.eval()

llama_preds = batched_generate_labels(
    llama_model,
    llama_tokenizer,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

llama_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), llama_preds)
print_metrics('LLaMA3.2-3B qLoRA (Default Hyperparams) on Gold Commodity', llama_metrics)

log_run_to_wandb(
    run_name='llama3.2-3b-qlora_default_fullfpb_gold',
    config_dict={
        'model_name': LLAMA_MODEL_NAME,
        'adapter_dir': LLAMA_ADAPTER_DIR,
        'stage': 'finetune_default',
        'dataset_train': 'financialphrasebank_75agree_full',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'epochs': 3,
        'lr': 2e-4,
        'batch_size': 8,
        'grad_accum': 4,
        'train_loss': float(train_out.training_loss),
    },
    metrics=llama_metrics,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_2161330/488260591.py:2: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Filter (num_proc=64): 100%|██████████| 3453/3453 [00:03<00:00, 1021.94 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,453 | Num Epochs = 3 | Total steps = 324
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,0.238700
40,0.077600
60,0.083900
80,0.077300
100,0.065200
120,0.047600
140,0.029900
160,0.025700
180,0.040700
200,0.019100


LLaMA3.2-3B qLoRA (Default Hyperparams) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.84      0.88      0.86       757
     neutral       0.12      0.67      0.20        82
    positive       0.86      0.83      0.85       881

   micro avg       0.69      0.84      0.76      1720
   macro avg       0.61      0.79      0.63      1720
weighted avg       0.82      0.84      0.82      1720

Macro F1   : 0.6345
Micro F1   : 0.7574
Weighted F1: 0.8212
Accuracy   : 0.6868
MCC        : 0.5754
F1 neg     : 0.8605
F1 neu     : 0.1975
F1 pos     : 0.8456
[W&B Disabled] Would have logged run: llama3.2-3b-qlora_default_fullfpb_gold


## 6. Random Search Tuning (Qwen qLoRA) + Final Evaluation on Gold

This stage uses a small random search over qLoRA hyperparameters with early stopping to keep tuning fast on GPU clusters.

In [10]:
import os
import gc
import random
import torch
from transformers import EarlyStoppingCallback

RANDOM_SEARCH_TRIALS = int(os.getenv("RANDOM_SEARCH_TRIALS", 2))
RANDOM_SEARCH_SEED = int(os.getenv("RANDOM_SEARCH_SEED", SEED))
rng = random.Random(RANDOM_SEARCH_SEED)

# much smaller search space
search_space = {
    "lora_r": [4, 8],
    "lora_alpha": [8, 16],
    "lora_dropout": [0.0, 0.05],
    "learning_rate": [1e-4, 2e-4],
    "batch_size": [1],
    "grad_accum": [8, 16],
    "epochs": [1],
}

def sample_random_params():
    return {
        "lora_r": rng.choice(search_space["lora_r"]),
        "lora_alpha": rng.choice(search_space["lora_alpha"]),
        "lora_dropout": rng.choice(search_space["lora_dropout"]),
        "learning_rate": rng.choice(search_space["learning_rate"]),
        "batch_size": 1,
        "grad_accum": rng.choice(search_space["grad_accum"]),
        "epochs": 1,
    }

if len(train_ds) == 0:
    raise ValueError("train_ds is empty")
if "text" not in train_ds.column_names:
    raise ValueError(f"train_ds missing 'text' column: {train_ds.column_names}")

# small subset first
train_ds_small = train_ds.select(range(min(200, len(train_ds))))
gold_sentences_small = gold_eval_df["sentence"].tolist()[:50]
gold_labels_small = gold_eval_df["label_text"].tolist()[:50]

best_random_params = None
best_random_score = -1.0
best_random_metrics = None
best_random_trial = -1

for trial_idx in range(RANDOM_SEARCH_TRIALS):
    params = sample_random_params()
    print(f"\nTrial {trial_idx + 1}/{RANDOM_SEARCH_TRIALS}: {params}")

    model = None
    tok = None
    trial_trainer = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    try:
        model, tok = FastLanguageModel.from_pretrained(
            model_name=QWEN_MODEL_NAME,
            max_seq_length=512,   # reduce here too
            dtype=None,
            load_in_4bit=True,
        )

        model = FastLanguageModel.get_peft_model(
            model,
            r=params["lora_r"],
            lora_alpha=params["lora_alpha"],
            lora_dropout=params["lora_dropout"],
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # attention only
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=SEED,
        )

        trial_out_dir = f"outputs/random_search/trial_{trial_idx}"

        trial_trainer = SFTTrainer(
            model=model,
            tokenizer=tok,
            train_dataset=train_ds_small,
            dataset_text_field="text",
            max_seq_length=512,
            data_collator=DataCollatorForSeq2Seq(tokenizer=tok),
            packing=False,
            args=SFTConfig(
                output_dir=trial_out_dir,
                num_train_epochs=1,
                per_device_train_batch_size=1,
                gradient_accumulation_steps=params["grad_accum"],
                learning_rate=params["learning_rate"],
                lr_scheduler_type="cosine",
                warmup_ratio=0.03,
                weight_decay=0.01,
                optim="adamw_8bit",
                logging_steps=20,
                save_strategy="no",
                eval_strategy="no",
                seed=SEED,
                report_to="none",
                fp16=not torch.cuda.is_bf16_supported(),
                bf16=torch.cuda.is_bf16_supported(),
                dataloader_pin_memory=False,
            ),
        )

        # catch OOM during TRAINING too
        trial_trainer.train()

        del trial_trainer
        trial_trainer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        FastLanguageModel.for_inference(model)
        model.eval()

        trial_preds = batched_generate_labels(
            model,
            tok,
            gold_sentences_small,
            batch_size=1,
            max_new_tokens=2,
        )
        trial_metrics = evaluate_predictions(gold_labels_small, trial_preds)

        print(f"Trial {trial_idx + 1} macro_f1: {trial_metrics['macro_f1']:.4f}")

        if trial_metrics["macro_f1"] > best_random_score:
            best_random_score = trial_metrics["macro_f1"]
            best_random_params = params
            best_random_metrics = trial_metrics
            best_random_trial = trial_idx + 1

    except torch.cuda.OutOfMemoryError:
        print(f"Trial {trial_idx + 1} OOM. Skipping.")
    except Exception as e:
        print(f"Trial {trial_idx + 1} failed: {type(e).__name__}: {e}")
    finally:
        if trial_trainer is not None:
            del trial_trainer
        if model is not None:
            del model
        if tok is not None:
            del tok
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

if best_random_params is None:
    print("No successful random-search trials. Run 1 fixed tiny trial first.")
else:
    print("\nBest random search trial:", best_random_trial)
    print("Best random search macro F1:", best_random_score)
    print("Best random search params:", best_random_params)
    print("Best random search metrics:", best_random_metrics)


Trial 1/2: {'lora_r': 4, 'lora_alpha': 8, 'lora_dropout': 0.05, 'learning_rate': 0.0001, 'batch_size': 1, 'grad_accum': 8, 'epochs': 1}
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Unsloth: Tokenizing ["text"] (num_proc=64): 100%|██████████| 200/200 [00:17<00:00, 11.71 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 1,843,200 of 3,087,781,888 (0.06% trained)


Step,Training Loss
20,3.072300


Trial 1 macro_f1: 0.5686

Trial 2/2: {'lora_r': 4, 'lora_alpha': 8, 'lora_dropout': 0.0, 'learning_rate': 0.0002, 'batch_size': 1, 'grad_accum': 8, 'epochs': 1}
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.4 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.
Unsloth: Tokenizing ["text"] (num_proc=64): 100%|██████████| 200/200 [00:17<00:00, 11.21 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 1,843,200 of 3,087,781,888 (0.06% trained)


Step,Training Loss
20,2.658900


Trial 2 macro_f1: 0.5881

Best random search trial: 2
Best random search macro F1: 0.5880587794007326
Best random search params: {'lora_r': 4, 'lora_alpha': 8, 'lora_dropout': 0.0, 'learning_rate': 0.0002, 'batch_size': 1, 'grad_accum': 8, 'epochs': 1}
Best random search metrics: {'macro_f1': 0.5880587794007326, 'micro_f1': 0.6813186813186813, 'weighted_f1': 0.74153832575791, 'f1_negative': 0.8837209302325582, 'f1_neutral': 0.23529411764705882, 'f1_positive': 0.6451612903225806, 'accuracy': 0.62, 'mcc': 0.5111216978940283, 'classification_report': '              precision    recall  f1-score   support\n\n    negative       0.83      0.95      0.88        20\n     neutral       0.13      1.00      0.24         2\n    positive       0.83      0.53      0.65        19\n\n   micro avg       0.62      0.76      0.68        41\n   macro avg       0.60      0.83      0.59        41\nweighted avg       0.80      0.76      0.74        41\n', 'confusion_matrix': array([[19,  0,  1],
       [ 0, 

In [ ]:
# Retrain one final model on full FPB with best random-search params, then evaluate/log
best = best_random_params
BEST_ADAPTER_DIR = 'qlora_adapters/qwen2.5-3b-fpb-random-search-best'

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

best_model, best_tok = FastLanguageModel.from_pretrained(
    model_name=QWEN_MODEL_NAME,
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

best_model = FastLanguageModel.get_peft_model(
    best_model,
    r=best['lora_r'],
    lora_alpha=best['lora_alpha'],
    lora_dropout=best['lora_dropout'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # lighter
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

best_trainer = SFTTrainer(
    model=best_model,
    tokenizer=best_tok,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=512,
    data_collator=DataCollatorForSeq2Seq(tokenizer=best_tok),
    packing=False,
    args=SFTConfig(
        output_dir='outputs/qwen2.5-3b-fpb-random-search-best',
        num_train_epochs=best['epochs'],
        per_device_train_batch_size=1,
        gradient_accumulation_steps=best['grad_accum'],
        learning_rate=best['learning_rate'],
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim='adamw_8bit',
        logging_steps=20,
        save_strategy='no',
        eval_strategy='no',
        seed=SEED,
        report_to='none',
        fp16=not torch.cuda.is_bf16_supported(), 
        bf16=torch.cuda.is_bf16_supported(),
        dataloader_pin_memory=False,
    ),
)

print('Starting final training with best random-search params...')
best_train_out = best_trainer.train()

best_model.save_pretrained(BEST_ADAPTER_DIR)
best_tok.save_pretrained(BEST_ADAPTER_DIR)

del best_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

FastLanguageModel.for_inference(best_model)
best_model.eval()

best_preds = batched_generate_labels(
    best_model,
    best_tok,
    gold_eval_df['sentence'].tolist(),
    batch_size=1,
    max_new_tokens=2,
)

best_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), best_preds)
print_metrics('Qwen2.5-3B qLoRA (Random Search Best) on Gold Commodity', best_metrics)

log_run_to_wandb(
    run_name='qwen2.5-3b-qlora_random_search_best_fullfpb_gold',
    config_dict={
        'model_name': QWEN_MODEL_NAME,
        'adapter_dir': BEST_ADAPTER_DIR,
        'stage': 'finetune_random_search_best',
        'dataset_train': 'financialphrasebank_75agree_full',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        **best,
        'train_loss': float(best_train_out.training_loss),
        'random_search_trial': int(best_random_trial),
        'random_search_macro_f1': float(best_random_score),
    },
    metrics=best_metrics,
)

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Tokenizing ["text"] (num_proc=64): 100%|██████████| 3453/3453 [00:18<00:00, 187.92 examples/s]


Starting final training with best random-search params...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,453 | Num Epochs = 1 | Total steps = 432
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 1,843,200 of 3,087,781,888 (0.06% trained)


Step,Training Loss
20,3.237400
40,1.269300
60,0.661200
80,0.633900
100,0.597700
120,0.577600
140,0.608700
160,0.557500
180,0.585600
200,0.541700


Qwen2.5-3B qLoRA (Random Search Best) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.86      0.72      0.79       757
     neutral       0.08      0.91      0.15        82
    positive       0.88      0.58      0.70       881

   micro avg       0.53      0.66      0.59      1720
   macro avg       0.61      0.74      0.55      1720
weighted avg       0.84      0.66      0.71      1720

Macro F1   : 0.5452
Micro F1   : 0.5889
Weighted F1: 0.7107
Accuracy   : 0.5341
MCC        : 0.4519
F1 neg     : 0.7862
F1 neu     : 0.1517
F1 pos     : 0.6978
[W&B Disabled] Would have logged run: qwen2.5-3b-qlora_random_search_best_fullfpb_gold


## 7. Compact Comparison Table

In [12]:
comparison_df = pd.DataFrame([
    {'model': 'Qwen2.5-3B-Instruct (zero-shot)', **{k: v for k, v in qwen_metrics.items() if isinstance(v, (int, float, np.floating))}},
    {'model': 'Qwen2.5-3B qLoRA (default)', **{k: v for k, v in llama_metrics.items() if isinstance(v, (int, float, np.floating))}},
    {'model': 'Qwen2.5-3B qLoRA (random search best)', **{k: v for k, v in best_metrics.items() if isinstance(v, (int, float, np.floating))}},
])

display_cols = ['model', 'macro_f1', 'micro_f1', 'weighted_f1', 'accuracy', 'mcc', 'f1_negative', 'f1_neutral', 'f1_positive']
display(comparison_df[display_cols].sort_values('macro_f1', ascending=False).reset_index(drop=True))

,model,macro_f1,micro_f1,weighted_f1,accuracy,mcc,f1_negative,f1_neutral,f1_positive
0,Qwen2.5-3B qLoRA (default),0.634509,0.757433,0.821231,0.686850,0.575366,0.860465,0.197487,0.845575
1,Qwen2.5-3B qLoRA (random search best),0.545216,0.588941,0.710661,0.534059,0.451937,0.786177,0.151668,0.697802
2,Qwen2.5-3B-Instruct (zero-shot),0.335690,0.347939,0.415292,0.315516,0.268387,0.693790,0.108757,0.204523


## 8. Placeholder Save of Best Model


In [13]:
from pathlib import Path

EXPORT_DIR = Path("experiments/sentiment_agent/saved_models/random_search_best_placeholder")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if "best_model" in globals() and "best_tok" in globals():
    best_model.save_pretrained(EXPORT_DIR)
    best_tok.save_pretrained(EXPORT_DIR)
    print(f"Saved best model/tokenizer to: {EXPORT_DIR}")
elif "llama_model" in globals() and "llama_tokenizer" in globals():
    llama_model.save_pretrained(EXPORT_DIR)
    llama_tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved fallback llama model/tokenizer to: {EXPORT_DIR}")
elif "baseline_model" in globals() and "baseline_tokenizer" in globals():
    baseline_model.save_pretrained(EXPORT_DIR)
    baseline_tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved baseline model/tokenizer to: {EXPORT_DIR}")
elif "model" in globals() and "tokenizer" in globals():
    model.save_pretrained(EXPORT_DIR)
    tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved generic model/tokenizer to: {EXPORT_DIR}")
else:
    placeholder = EXPORT_DIR / "README_PLACEHOLDER.txt"
    placeholder.write_text(
        "No in-memory model object found. This directory is reserved for the best model export.",
        encoding="utf-8",
    )
    print(f"Created placeholder file: {placeholder}")

Saved best model/tokenizer to: experiments/sentiment_agent/saved_models/random_search_best_placeholder


## 8. Cleanup (Optional)

In [14]:
for var_name in ['qwen_model', 'qwen_tokenizer', 'llama_model', 'llama_tokenizer', 'best_model', 'best_tok', 'trainer', 'best_trainer']:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleanup complete.')

Cleanup complete.
